Supplementary Code S11
Nested-CV Comparison of Random Forest and Simple Baseline Models
Purpose

This notebook implements a reviewer-requested robustness analysis comparing the primary non-linear model family (Random Forest, RF) against simpler baseline classifiers—Logistic Regression (LR) and k-Nearest Neighbors (k-NN)—under the same, strictly nested cross-validation (CV) protocol used throughout the study.

The goal is to assess whether the near-perfect discrimination observed in some variants reflects intrinsic separability of the data, rather than being driven solely by model complexity.

Rationale

If simple models (LR, k-NN), which have limited representational capacity, achieve similarly high outer-fold AUCs as RF, this supports the interpretation that:

the predictive signal is strong and robust,

performance is not an artifact of flexible model architectures.

Conversely, large performance gaps between RF and simpler models would indicate that model complexity is critical for capturing non-linear interactions.

Methodological safeguards

All analyses use properly nested CV:

Outer CV provides unbiased performance estimates.

Inner CV is used exclusively for hyperparameter tuning.

All preprocessing steps (median imputation, scaling) are embedded in the pipeline and refit within each training fold only.

No information from outer test folds is used in:

feature handling,

scaling,

model fitting,

or hyperparameter selection.

Feature matrices are constructed using the same governed feature sets defined in earlier Supplementary Codes (S2, S6–S9), with:

automatic removal of all-NaN and constant predictors,

consistent target definition.

Inputs

Required:

processed_dataset.csv

feature_variants.json

Optional (recommended for reproducibility diagnostics):

audit_nonnumeric_features.csv

Models evaluated

Logistic Regression (LR)
Regularized linear baseline (L1/L2/elastic-net explored via inner CV).

k-Nearest Neighbors (k-NN)
Non-parametric distance-based baseline.

Random Forest (RF)
Non-linear ensemble model aligned with the primary analyses.

All models are evaluated separately for each feature-set variant (FULL, CLINICAL, BIOMARKERS).

Outputs

S11_nestedcv_simple_models_results.csv
Summary table reporting mean ± SD of outer-fold AUC and MCC for each
model × feature-variant combination.

S11_nestedcv_simple_models_folds.csv
Per-fold AUC/MCC values and selected hyperparameters (complete audit trail).

S11_nestedcv_simple_models_perfect_auc_flags.csv
Counts of near-perfect discrimination across outer folds
(AUC ≥ 0.99 and AUC = 1.0).

Notes

AUC is computed from predicted probabilities on outer test folds only.

MCC is reported at a fixed threshold of 0.5 and is descriptive, not optimized;
threshold selection is handled separately where required (e.g., S4, S9).

Feature scaling is included for all models for pipeline consistency;
Random Forest is invariant to monotonic scaling, so this does not introduce bias.

This analysis is diagnostic and confirmatory, not intended for final model selection.

## Setup + catalogs

In [1]:
from __future__ import annotations

import json
import warnings
from datetime import datetime
from pathlib import Path
from typing import Dict, Any, Tuple, List

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import roc_auc_score, matthews_corrcoef

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*Skipping features without any observed values.*")

ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
OUT_DIR = ROOT / "results" / "S11_simple_models_nestedcv"
TAB_DIR = OUT_DIR / "tables"
LOG_DIR = OUT_DIR / "logs"
for d in (TAB_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("Outputs:", OUT_DIR.resolve())


ROOT: /content
Outputs: /content/results/S11_simple_models_nestedcv


## Finder + loadery

In [2]:
SEARCH_ROOTS = [
    ROOT,
    ROOT / "results",
    ROOT / "results" / "S9_locked_test",
    ROOT / "results" / "S6_leakage_audit",
    ROOT / "results" / "S8_revised_models",
    Path("/mnt/data"),
    ROOT / "drive",
    ROOT / "drive" / "MyDrive",
]

def find_file(filename: str, roots: list[Path]) -> Path:
    for r in roots:
        p = r / filename
        if p.exists():
            return p
    for r in roots:
        if r.exists():
            hits = list(r.rglob(filename))
            if hits:
                hits = sorted(hits, key=lambda x: len(str(x)))
                return hits[0]
    raise FileNotFoundError(
        f"Missing required file: {filename}\nSearched in:\n - " + "\n - ".join(map(str, roots))
    )

def read_csv_required(name: str) -> tuple[pd.DataFrame, Path]:
    p = find_file(name, SEARCH_ROOTS)
    df = pd.read_csv(p)
    print(f"Loaded: {p} | shape={df.shape}")
    return df, p

def load_feature_list(path: Path) -> list[str]:
    fdf = pd.read_csv(path)
    fdf.columns = [c.strip() for c in fdf.columns]
    for col in ["feature","features","feature_name","column","variable","var","name"]:
        if col in fdf.columns:
            feats = fdf[col].astype(str).str.strip().tolist()
            return [f for f in feats if f and f.lower() != "nan"]
    feats = fdf.iloc[:, 0].astype(str).str.strip().tolist()
    return [f for f in feats if f and f.lower() != "nan"]


## Configuration CV

In [3]:
N_OUTER = 5
N_INNER = 5

cv_outer = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)
cv_inner = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)

print("Nested CV:", f"outer={N_OUTER}, inner={N_INNER}, RANDOM_STATE={RANDOM_STATE}")


Nested CV: outer=5, inner=5, RANDOM_STATE=42


## Data + governed features_used_*.csv

In [4]:
df, path_processed = read_csv_required("processed_dataset.csv")

p_full = find_file("features_used_FULL.csv", SEARCH_ROOTS)
p_clin = find_file("features_used_CLINICAL.csv", SEARCH_ROOTS)
p_bio  = find_file("features_used_BIOMARKERS.csv", SEARCH_ROOTS)

features_used = {
    "FULL": load_feature_list(p_full),
    "CLINICAL": load_feature_list(p_clin),
    "BIOMARKERS": load_feature_list(p_bio),
}

print("Governed feature counts:", {k: len(v) for k, v in features_used.items()})


Loaded: /content/processed_dataset.csv | shape=(152, 117)
Governed feature counts: {'FULL': 69, 'CLINICAL': 58, 'BIOMARKERS': 10}


## Target

In [5]:
TARGET_COL = "typ_zawalu"
if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in processed_dataset.csv")

allowed = {"STEMI", "NSTEMI"}
df = df.loc[df[TARGET_COL].astype(str).str.strip().isin(allowed)].copy()
y = (df[TARGET_COL].astype(str).str.strip() == "STEMI").astype(int).values

for k in features_used:
    features_used[k] = [f for f in features_used[k] if f != TARGET_COL]

print("TARGET_COL:", TARGET_COL)
print("Task: STEMI vs NSTEMI (positive=STEMI)")
print(pd.Series(y).value_counts())
print("Positive rate:", float(np.mean(y)))
print("N samples:", len(y))


TARGET_COL: typ_zawalu
Task: STEMI vs NSTEMI (positive=STEMI)
0    32
1    25
Name: count, dtype: int64
Positive rate: 0.43859649122807015
N samples: 57


## Optional audit_nonnumeric + matrix construction (drop all-NaN/constant)

In [6]:
NONNUMERIC_FEATURES = set()
try:
    audit_df, path_audit = read_csv_required("audit_nonnumeric_features.csv")
    cand = ["feature","feature_name","column","variable","var","name"]
    col = next((c for c in cand if c in audit_df.columns), None)
    if col is None:
        col = audit_df.columns[0]
    NONNUMERIC_FEATURES = set(audit_df[col].astype(str).tolist())
    print("Loaded NONNUMERIC_FEATURES:", len(NONNUMERIC_FEATURES))
except FileNotFoundError:
    path_audit = None
    print("NOTE: audit_nonnumeric_features.csv not found (OK).")

YES_SET = {"yes","y","true","t","1","tak","on","present","positive","pos"}
NO_SET  = {"no","n","false","f","0","nie","off","absent","negative","neg"}

def _normalize_str_series(s: pd.Series) -> pd.Series:
    return (s.astype(str)
             .str.strip()
             .str.lower()
             .str.replace(",", ".", regex=False))

def _try_map_binary_object(col: pd.Series) -> tuple[pd.Series, bool]:
    norm = _normalize_str_series(col)
    uniq = sorted(norm.dropna().unique().tolist())
    if len(uniq) == 0:
        return col, False
    if len(set(uniq)) <= 2 and all(u in (YES_SET | NO_SET) for u in uniq):
        def map_one(v):
            if pd.isna(v): return np.nan
            vv = str(v).strip().lower()
            if vv in YES_SET: return 1
            if vv in NO_SET:  return 0
            return np.nan
        return norm.map(map_one).astype(float), True
    return col, False

def build_matrix(df_in: pd.DataFrame, feats: list[str], variant_name: str):
    feats = [f for f in feats if f in df_in.columns and f != TARGET_COL]
    feats = [f for f in feats if f not in NONNUMERIC_FEATURES]

    Xdf = df_in[feats].copy()

    recoded = 0
    for c in Xdf.columns:
        if Xdf[c].dtype == "object" or str(Xdf[c].dtype).startswith("bool"):
            mapped, ok = _try_map_binary_object(Xdf[c])
            if ok:
                Xdf[c] = mapped
                recoded += 1

    for c in Xdf.columns:
        if Xdf[c].dtype == "object":
            Xdf[c] = _normalize_str_series(Xdf[c])
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce")

    all_nan = Xdf.columns[Xdf.isna().mean() == 1.0].tolist()
    constant = Xdf.columns[Xdf.nunique(dropna=True) <= 1].tolist()
    dropped = sorted(set(all_nan + constant))

    if dropped:
        pd.DataFrame({"feature": dropped}).to_csv(
            TAB_DIR / f"S11_{variant_name}_dropped_empty_or_constant.csv", index=False
        )
        Xdf = Xdf.drop(columns=dropped)

    diag = {
        "variant": variant_name,
        "n_governed_present": int(len(feats)),
        "n_recoded_binary": int(recoded),
        "n_dropped_empty_or_constant": int(len(dropped)),
        "n_used_matrix": int(Xdf.shape[1]),
    }
    return Xdf.values, np.array(Xdf.columns.tolist()), diag

variants: Dict[str, Tuple[np.ndarray, np.ndarray]] = {}
diag_rows = []
for v in ["FULL","CLINICAL","BIOMARKERS"]:
    Xv, fn, d = build_matrix(df, features_used[v], v)
    variants[v] = (Xv, fn)
    diag_rows.append(d)

diag_df = pd.DataFrame(diag_rows).sort_values("variant").reset_index(drop=True)
diag_df.to_csv(TAB_DIR / "S11_variant_matrix_diagnostics.csv", index=False)

display(diag_df)

bad = diag_df[diag_df["n_used_matrix"] < 2]
if len(bad) > 0:
    raise RuntimeError("S11 cannot run: some variants have <2 usable predictors.\n" + bad.to_string(index=False))


NOTE: audit_nonnumeric_features.csv not found (OK).


,variant,n_governed_present,n_recoded_binary,n_dropped_empty_or_constant,n_used_matrix
0,BIOMARKERS,10,0,0,10
1,CLINICAL,58,20,16,42
2,FULL,69,20,16,53


## Models + grid (without invalid combos) + model-aware preprocessing

### Interpretation note — why we benchmark *simple models* (and why AUC≈1.0 is a warning)

This notebook performs **workflow-level diagnostic validation** (manuscript Section 5.7) using:
1) **permutation testing** (label shuffling), and  
2) **simplified / low-capacity baseline models** (e.g., penalized logistic regression).

**Key point:** if a very simple model achieves **AUC ≈ 1.0** under internal evaluation (most often in the permissive **FULL** variant), this is treated as a **diagnostic red flag** suggesting potential leakage, structural artifacts, or downstream-coupled predictors — *not* as a success criterion.

Therefore, perfect or near-perfect AUC values in this notebook should be interpreted as *signals to investigate feature provenance and governance*, consistent with the leakage-aware workflow described in the manuscript.


In [7]:
MODELS: Dict[str, Any] = {
    "LR_L2": LogisticRegression(
        penalty="l2", solver="lbfgs", max_iter=8000, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "LR_L1": LogisticRegression(
        penalty="l1", solver="saga", max_iter=8000, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "LR_ELNET": LogisticRegression(
        penalty="elasticnet", solver="saga", max_iter=8000, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "KNN": KNeighborsClassifier(),
    "RF": RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
}

GRIDS: Dict[str, Dict[str, Any]] = {
    "LR_L2": {"clf__C": np.logspace(-3, 2, 10)},
    "LR_L1": {"clf__C": np.logspace(-3, 2, 10)},
    "LR_ELNET": {"clf__C": np.logspace(-3, 2, 10), "clf__l1_ratio": [0.1, 0.5, 0.9]},
    "KNN": {
        "clf__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
        "clf__weights": ["uniform", "distance"],
        "clf__p": [1, 2],
    },
    "RF": {
        "clf__n_estimators": [300, 600],
        "clf__max_depth": [2, 3, 4, 5, 8, None],
        "clf__min_samples_leaf": [1, 2, 5, 10],
        "clf__max_features": ["sqrt", 0.5],
    },
}

def make_pipeline(model_key: str) -> Pipeline:

    pre_steps = [("imputer", SimpleImputer(strategy="median"))]
    if model_key.startswith("LR_"):
        pre_steps.append(("scaler", StandardScaler()))
    preprocess = Pipeline(pre_steps)

    return Pipeline([
        ("preprocess", preprocess),
        ("clf", MODELS[model_key]),
    ])

print("Models:", list(MODELS.keys()))


Models: ['LR_L2', 'LR_L1', 'LR_ELNET', 'KNN', 'RF']


## Nested CV eval

In [8]:
def nested_cv_eval(X: np.ndarray, y: np.ndarray, variant_name: str, model_key: str) -> Tuple[pd.DataFrame, Dict[str, float]]:
    aucs: List[float] = []
    mccs: List[float] = []
    rows_folds: List[Dict[str, Any]] = []

    for fold_id, (tr, te) in enumerate(cv_outer.split(X, y), start=1):
        X_tr, X_te = X[tr], X[te]
        y_tr, y_te = y[tr], y[te]

        pipe = make_pipeline(model_key)

        gs = GridSearchCV(
            estimator=pipe,
            param_grid=GRIDS[model_key],
            scoring="roc_auc",
            cv=cv_inner,
            n_jobs=-1,
            refit=True,
            error_score="raise",
        )
        gs.fit(X_tr, y_tr)

        best_model = gs.best_estimator_
        probs = best_model.predict_proba(X_te)[:, 1]

        auc = float(roc_auc_score(y_te, probs))
        mcc = float(matthews_corrcoef(y_te, (probs >= 0.5).astype(int)))

        aucs.append(auc)
        mccs.append(mcc)

        row = {
            "variant": variant_name,
            "model": model_key,
            "outer_fold": fold_id,
            "auc": auc,
            "mcc@0.5": mcc,
            "best_inner_auc": float(gs.best_score_),
        }
        for k, v in gs.best_params_.items():
            row[k] = v
        rows_folds.append(row)

    df_folds = pd.DataFrame(rows_folds)

    summary = {
        "auc_mean": float(np.mean(aucs)),
        "auc_sd": float(np.std(aucs, ddof=1)),
        "mcc_mean": float(np.mean(mccs)),
        "mcc_sd": float(np.std(mccs, ddof=1)),
    }
    return df_folds, summary

def count_near_perfect(aucs: np.ndarray, thr: float = 0.99) -> Dict[str, int]:
    return {
        "n_folds": int(len(aucs)),
        "n_auc_ge_thr": int(np.sum(aucs >= thr)),
        "n_auc_eq_1.0": int(np.sum(np.isclose(aucs, 1.0))),
    }


## Launch + registration

In [9]:
rows_summary: List[Dict[str, Any]] = []
rows_folds_all: List[pd.DataFrame] = []
rows_perfect: List[Dict[str, Any]] = []

for variant_name, (Xv, feats_v) in variants.items():
    print("\n==============================")
    print("S11 Nested CV:", variant_name)
    print("==============================")
    print("X:", Xv.shape, "n_features:", len(feats_v))

    for model_key in ["RF", "LR_L2", "LR_L1", "LR_ELNET", "KNN"]:
        print(f"\n--- Model: {model_key} ---")
        df_folds, summary = nested_cv_eval(Xv, y, variant_name=variant_name, model_key=model_key)

        rows_folds_all.append(df_folds)
        rows_summary.append({
            "variant": variant_name,
            "model": model_key,
            **summary,
            "n_samples": int(Xv.shape[0]),
            "n_features": int(Xv.shape[1]),
        })

        perf = count_near_perfect(df_folds["auc"].values, thr=0.99)
        rows_perfect.append({"variant": variant_name, "model": model_key, **perf})

        print(f"AUC (mean±SD): {summary['auc_mean']:.3f} ± {summary['auc_sd']:.3f}")
        print(f"MCC@0.5 (mean±SD): {summary['mcc_mean']:.3f} ± {summary['mcc_sd']:.3f}")
        print(f"Near-perfect: {perf['n_auc_ge_thr']}/{perf['n_folds']} folds (AUC>=0.99), "
              f"{perf['n_auc_eq_1.0']} folds (AUC==1.0)")

df_summary = pd.DataFrame(rows_summary).sort_values(["variant", "auc_mean"], ascending=[True, False]).reset_index(drop=True)
df_folds_all = pd.concat(rows_folds_all, ignore_index=True)
df_perfect = pd.DataFrame(rows_perfect).sort_values(["variant", "model"]).reset_index(drop=True)

p1 = TAB_DIR / "S11_nestedcv_simple_models_results.csv"
p2 = TAB_DIR / "S11_nestedcv_simple_models_folds.csv"
p3 = TAB_DIR / "S11_nestedcv_simple_models_perfect_auc_flags.csv"

df_summary.to_csv(p1, index=False)
df_folds_all.to_csv(p2, index=False)
df_perfect.to_csv(p3, index=False)

print("\nS11 completed. Outputs:")
print(" -", p1)
print(" -", p2)
print(" -", p3)

display(df_summary)



S11 Nested CV: FULL
X: (57, 53) n_features: 53

--- Model: RF ---
AUC (mean±SD): 0.994 ± 0.013
MCC@0.5 (mean±SD): 0.931 ± 0.153
Near-perfect: 4/5 folds (AUC>=0.99), 4 folds (AUC==1.0)

--- Model: LR_L2 ---
AUC (mean±SD): 0.901 ± 0.130
MCC@0.5 (mean±SD): 0.732 ± 0.286
Near-perfect: 2/5 folds (AUC>=0.99), 2 folds (AUC==1.0)

--- Model: LR_L1 ---
AUC (mean±SD): 1.000 ± 0.000
MCC@0.5 (mean±SD): 0.969 ± 0.069
Near-perfect: 5/5 folds (AUC>=0.99), 5 folds (AUC==1.0)

--- Model: LR_ELNET ---
AUC (mean±SD): 1.000 ± 0.000
MCC@0.5 (mean±SD): 0.943 ± 0.128
Near-perfect: 5/5 folds (AUC>=0.99), 5 folds (AUC==1.0)

--- Model: KNN ---
AUC (mean±SD): 1.000 ± 0.000
MCC@0.5 (mean±SD): 0.969 ± 0.069
Near-perfect: 5/5 folds (AUC>=0.99), 5 folds (AUC==1.0)

S11 Nested CV: CLINICAL
X: (57, 42) n_features: 42

--- Model: RF ---
AUC (mean±SD): 0.582 ± 0.157
MCC@0.5 (mean±SD): 0.080 ± 0.364
Near-perfect: 0/5 folds (AUC>=0.99), 0 folds (AUC==1.0)

--- Model: LR_L2 ---
AUC (mean±SD): 0.479 ± 0.083
MCC@0.5 (mean±

,variant,model,auc_mean,auc_sd,mcc_mean,mcc_sd,n_samples,n_features
0,BIOMARKERS,RF,0.949524,0.050575,0.755713,0.163851,57,10
1,BIOMARKERS,KNN,0.908571,0.083869,0.654486,0.093686,57,10
2,BIOMARKERS,LR_L1,0.849524,0.088743,0.622060,0.228895,57,10
3,BIOMARKERS,LR_ELNET,0.820952,0.036359,0.513597,0.208059,57,10
4,BIOMARKERS,LR_L2,0.814286,0.135944,0.455624,0.310946,57,10
5,CLINICAL,RF,0.581905,0.157482,0.080367,0.363817,57,42
6,CLINICAL,LR_ELNET,0.505714,0.061112,-0.026461,0.137269,57,42
7,CLINICAL,KNN,0.501905,0.149663,-0.037396,0.244377,57,42
8,CLINICAL,LR_L2,0.479048,0.082589,0.111920,0.130051,57,42
9,CLINICAL,LR_L1,0.472857,0.090125,0.006199,0.063644,57,42


## Run log

In [10]:
run_log = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "random_state": RANDOM_STATE,
    "target": {
        "TARGET_COL": TARGET_COL,
        "task": "STEMI vs NSTEMI (positive=STEMI)",
        "n_samples": int(len(y)),
        "positive_rate": float(np.mean(y)),
    },
    "cv": {"outer": N_OUTER, "inner": N_INNER},
    "inputs": {
        "processed_dataset.csv": str(path_processed),
        "features_used_FULL.csv": str(p_full),
        "features_used_CLINICAL.csv": str(p_clin),
        "features_used_BIOMARKERS.csv": str(p_bio),
        "audit_nonnumeric_features.csv": str(path_audit) if path_audit else None,
    },
    "outputs": {
        "S11_variant_matrix_diagnostics.csv": str(TAB_DIR / "S11_variant_matrix_diagnostics.csv"),
        "S11_nestedcv_simple_models_results.csv": str(p1),
        "S11_nestedcv_simple_models_folds.csv": str(p2),
        "S11_nestedcv_simple_models_perfect_auc_flags.csv": str(p3),
    },
    "notes": [
        "Governed feature lists are taken from features_used_<VARIANT>.csv to ensure consistency with S5–S9.",
        "All-NaN and constant predictors are dropped before CV to avoid imputer warnings and unstable feature counts.",
        "Scaling is applied only for linear models (LR_*), not for RF/KNN (model-aware preprocessing).",
    ],
}

log_path = LOG_DIR / "S11_run_log.json"
log_path.write_text(json.dumps(run_log, indent=2), encoding="utf-8")
print("Saved:", log_path)


Saved: /content/results/S11_simple_models_nestedcv/logs/S11_run_log.json


/tmp/ipython-input-4250148233.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",
